# 🔍 Analisis Exploratorio y Limpieza — INUMET
**Materia:** Herramientas de Software para Big Data  
**Fuente:** Instituto Uruguayo de Meteorologia  

**Flujo de zonas:**
```
/lnd  →  datos crudos tal cual llegan de NiFi
/raw  →  esquema corregido y tipos validados
/rfn  →  datos limpios: nulos, duplicados y reglas de negocio aplicadas
```

**Tablas:**
| Archivo | Columnas |
|---------|----------|
| inumet_temperatura_del_aire.csv | fecha, estacion_id, temp_aire (°C) |
| inumet_intensidad_de_viento.csv | fecha, estacion_id, dir_viento (°), int_viento (km/h) |
| inumet_precipitacion_acumulada_horaria.csv | fecha, estacion_id, precip_horaria (mm) |
| inumet_humedad_relativa.csv | fecha, estacion_id, [ver printSchema] |
| inumet_presion_atmosferica_a_nive_del_mar.csv | fecha, estacion_id, [ver printSchema] |
| inumet_heliofania.csv | fecha, estacion_id, [ver printSchema] |

## 1. Inicializacion de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnull, isnan,
    to_timestamp, year, month, hour,
    avg, min, max, round as spark_round,
    trim, upper
)
from pyspark.sql.types import DoubleType, TimestampType, StringType

spark = SparkSession.builder \
    .appName("INUMET_EDA_Limpieza") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo")

## 2. Carga desde /lnd (datos crudos)

In [ ]:
LND = "hdfs://localhost:9000/lnd/obligatorio"

# Lectura cruda — sin inferSchema para ver exactamente como llegan los datos
df_temp    = spark.read.csv(f"{LND}/inumet_temperatura_del_aire.csv",              header=True, sep=";")
df_viento  = spark.read.csv(f"{LND}/inumet_intensidad_de_viento.csv",              header=True, sep=";")
df_lluvia  = spark.read.csv(f"{LND}/inumet_precipitacion_acumulada_horaria.csv",   header=True, sep=";")
df_humedad = spark.read.csv(f"{LND}/inumet_humedad_relativa.csv",                  header=True, sep=";")
df_presion = spark.read.csv(f"{LND}/inumet_presion_atmosferica_a_nive_del_mar.csv",header=True, sep=";")
df_helio   = spark.read.csv(f"{LND}/inumet_heliofania.csv",                        header=True, sep=";")

tablas = {
    "temperatura":    df_temp,
    "viento":         df_viento,
    "precipitacion":  df_lluvia,
    "humedad":        df_humedad,
    "presion":        df_presion,
    "heliofania":     df_helio,
}

print("Registros por tabla:")
for nombre, df in tablas.items():
    print(f"  {nombre:<15}: {df.count():>10,}")

## 3. Vista previa y schema de cada tabla
> En esta celda se identifican los nombres reales de columnas de humedad, presion y heliofania.

In [ ]:
for nombre, df in tablas.items():
    print(f"\n{'='*55}")
    print(f" TABLA: {nombre.upper()}")
    print(f" Columnas ({len(df.columns)}): {df.columns}")
    print(f"{'='*55}")
    df.printSchema()
    df.show(3, truncate=False)

## 4. Descripcion de columnas por tabla
Significado de cada columna dentro del dominio meteorologico.

In [ ]:
descripcion = {
    "temperatura": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        "temp_aire":  "Temperatura del aire en grados Celsius (°C)"
    },
    "viento": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        "dir_viento": "Direccion del viento en grados sexagesimales (0-360°)",
        "int_viento": "Intensidad del viento en kilometros por hora (km/h)"
    },
    "precipitacion": {
        "fecha":          "Fecha y hora de la medicion en UTC",
        "estacion_id":    "Identificador de la estacion meteorologica",
        "precip_horaria": "Precipitacion acumulada en el intervalo de una hora (mm)"
    },
    "humedad": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        # Actualizar con el nombre real visto en printSchema
        "col_3":      "Humedad relativa del aire expresada en porcentaje (%)"
    },
    "presion": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        # Actualizar con el nombre real visto en printSchema
        "col_3":      "Presion atmosferica al nivel del mar en hectopascales (hPa)"
    },
    "heliofania": {
        "fecha":      "Fecha y hora de la medicion en UTC",
        "estacion_id":"Identificador de la estacion meteorologica",
        # Actualizar con el nombre real visto en printSchema
        "col_3":      "Horas de insolacion solar registradas en el intervalo horario"
    },
}

for tabla, cols in descripcion.items():
    print(f"\n{tabla.upper()}")
    print(f"{'Columna':<20} {'Descripcion'}")
    print("-"*60)
    for col_name, desc in cols.items():
        print(f"  {col_name:<18} {desc}")

## 5. Correccion de tipos y guardado en /raw
En `/lnd` todo llega como `String`. Aqui se castean los tipos correctos y se guarda en `/raw`.

In [ ]:
# Actualizar col_valor_humedad, col_valor_presion, col_valor_helio
# con los nombres reales vistos en la celda 3
col_valor_humedad = "humedad_relativa"   # <-- ajustar si es diferente
col_valor_presion = "presion_atmosferica" # <-- ajustar si es diferente
col_valor_helio   = "heliofania"          # <-- ajustar si es diferente

def castear_tipos(df, col_numericas):
    """Castea fecha a timestamp y columnas numericas a Double."""
    df = df.withColumn("fecha", to_timestamp(col("fecha")))
    for c in col_numericas:
        df = df.withColumn(c, col(c).cast(DoubleType()))
    return df

df_temp_raw    = castear_tipos(df_temp,    ["temp_aire"])
df_viento_raw  = castear_tipos(df_viento,  ["dir_viento", "int_viento"])
df_lluvia_raw  = castear_tipos(df_lluvia,  ["precip_horaria"])
df_humedad_raw = castear_tipos(df_humedad, [col_valor_humedad])
df_presion_raw = castear_tipos(df_presion, [col_valor_presion])
df_helio_raw   = castear_tipos(df_helio,   [col_valor_helio])

print("Schemas corregidos:")
df_temp_raw.printSchema()
df_viento_raw.printSchema()

In [ ]:
# Guardar en /raw como parquet (mas eficiente que CSV para Spark)
RAW = "hdfs://localhost:9000/raw/obligatorio"

df_temp_raw.write.mode("overwrite").parquet(f"{RAW}/temperatura")
df_viento_raw.write.mode("overwrite").parquet(f"{RAW}/viento")
df_lluvia_raw.write.mode("overwrite").parquet(f"{RAW}/precipitacion")
df_humedad_raw.write.mode("overwrite").parquet(f"{RAW}/humedad")
df_presion_raw.write.mode("overwrite").parquet(f"{RAW}/presion")
df_helio_raw.write.mode("overwrite").parquet(f"{RAW}/heliofania")

print("Datos guardados en /raw correctamente")

## 6. Analisis de valores nulos

In [ ]:
def reporte_nulos(df, nombre):
    total = df.count()
    print(f"\n{'='*50}")
    print(f" {nombre} — {total:,} registros")
    print(f" {'Columna':<22} {'Nulos':>8} {'%':>8}")
    print("-"*42)
    for c in df.columns:
        nulos = df.filter(isnull(col(c))).count()
        pct   = nulos / total * 100
        icono = "⚠" if nulos > 0 else "✓"
        print(f" {icono} {c:<20} {nulos:>8,} {pct:>7.2f}%")

tablas_raw = {
    "TEMPERATURA":   df_temp_raw,
    "VIENTO":        df_viento_raw,
    "PRECIPITACION": df_lluvia_raw,
    "HUMEDAD":       df_humedad_raw,
    "PRESION":       df_presion_raw,
    "HELIOFANIA":    df_helio_raw,
}

for nombre, df in tablas_raw.items():
    reporte_nulos(df, nombre)

## 7. Analisis de duplicados

In [ ]:
def reporte_duplicados(df, nombre, claves):
    total    = df.count()
    distintos = df.dropDuplicates().count()
    duplicados = total - distintos
    print(f"\n{nombre}")
    print(f"  Total registros:      {total:>10,}")
    print(f"  Registros unicos:     {distintos:>10,}")
    print(f"  Duplicados exactos:   {duplicados:>10,}")
    
    # Duplicados por clave primaria (fecha + estacion_id)
    dup_clave = total - df.dropDuplicates(claves).count()
    print(f"  Duplicados por clave ({claves}): {dup_clave:,}")

for nombre, df in tablas_raw.items():
    reporte_duplicados(df, nombre, ["fecha", "estacion_id"])

## 8. Claves primarias unicas
La clave primaria de cada tabla es `(fecha, estacion_id)`. Verificamos que sea unica.

In [ ]:
print("Verificacion de clave primaria (fecha + estacion_id):\n")
print(f"{'Tabla':<15} {'Total':>10} {'Claves unicas':>15} {'Unicidad'}")
print("-"*52)

for nombre, df in tablas_raw.items():
    total  = df.count()
    unicos = df.select("fecha", "estacion_id").distinct().count()
    ok     = "✅ OK" if total == unicos else "❌ HAY DUPLICADOS"
    print(f"{nombre:<15} {total:>10,} {unicos:>15,} {ok}")

## 9. Reglas de negocio
Rangos validos para variables meteorologicas en Uruguay segun dominio.

In [ ]:
print("Reglas de negocio definidas para el dominio meteorologico:\n")
reglas = [
    ("temp_aire",        "-20",  "50",  "Temperatura del aire en Uruguay (°C)"),
    ("int_viento",       "0",    "200", "Intensidad de viento, max historico en Uruguay (km/h)"),
    ("dir_viento",       "0",    "360", "Direccion del viento en grados sexagesimales"),
    ("precip_horaria",   "0",    "300", "Precipitacion horaria, max historico en Uruguay (mm)"),
    (col_valor_humedad,  "0",    "100", "Humedad relativa, rango fisicamente posible (%)"),
    (col_valor_presion,  "900",  "1100","Presion atmosferica al nivel del mar (hPa)"),
    (col_valor_helio,    "0",    "24",  "Horas de insolacion, max posible en un dia"),
]
print(f"{'Columna':<25} {'Min':>6} {'Max':>6}  Descripcion")
print("-"*75)
for col_n, mn, mx, desc in reglas:
    print(f"  {col_n:<23} {mn:>6} {mx:>6}  {desc}")

In [ ]:
# Verificar cuantos registros violan las reglas de negocio
print("Registros fuera de rango:\n")

checks = [
    (df_temp_raw,    "temperatura",   "temp_aire",       -20,  50),
    (df_viento_raw,  "viento",        "int_viento",        0, 200),
    (df_viento_raw,  "viento",        "dir_viento",        0, 360),
    (df_lluvia_raw,  "precipitacion", "precip_horaria",    0, 300),
    (df_humedad_raw, "humedad",       col_valor_humedad,   0, 100),
    (df_presion_raw, "presion",       col_valor_presion,  900,1100),
    (df_helio_raw,   "heliofania",    col_valor_helio,     0,  24),
]

for df, tabla, columna, mn, mx in checks:
    fuera = df.filter(
        col(columna).isNotNull() & ((col(columna) < mn) | (col(columna) > mx))
    ).count()
    icono = "⚠" if fuera > 0 else "✓"
    print(f"  {icono} {tabla:<15} {columna:<25} fuera de rango: {fuera:,}")

## 10. Limpieza y guardado en /rfn
Se aplican las siguientes transformaciones:
- Eliminar registros con nulos en columnas clave
- Eliminar duplicados exactos
- Eliminar registros que violan las reglas de negocio
- Limpiar espacios en `estacion_id`

In [ ]:
def limpiar(df, col_valor, min_val, max_val):
    """Aplica limpieza completa a un DataFrame de INUMET."""
    total_original = df.count()
    
    # 1. Limpiar espacios en estacion_id
    df = df.withColumn("estacion_id", trim(col("estacion_id")))
    
    # 2. Eliminar nulos en columnas clave
    df = df.filter(
        col("fecha").isNotNull() &
        col("estacion_id").isNotNull() &
        col(col_valor).isNotNull()
    )
    tras_nulos = df.count()
    
    # 3. Eliminar duplicados exactos
    df = df.dropDuplicates()
    tras_duplicados = df.count()
    
    # 4. Eliminar registros fuera de rango (reglas de negocio)
    df = df.filter((col(col_valor) >= min_val) & (col(col_valor) <= max_val))
    tras_reglas = df.count()
    
    print(f"  Original:              {total_original:>10,}")
    print(f"  Tras eliminar nulos:   {tras_nulos:>10,}  (-{total_original - tras_nulos:,})")
    print(f"  Tras eliminar dupl.:   {tras_duplicados:>10,}  (-{tras_nulos - tras_duplicados:,})")
    print(f"  Tras reglas negocio:   {tras_reglas:>10,}  (-{tras_duplicados - tras_reglas:,})")
    print(f"  Registros finales:     {tras_reglas:>10,}")
    
    return df

print("=== LIMPIEZA - TEMPERATURA ===")
df_temp_rfn = limpiar(df_temp_raw, "temp_aire", -20, 50)

print("\n=== LIMPIEZA - VIENTO ===")
df_viento_rfn = df_viento_raw \
    .withColumn("estacion_id", trim(col("estacion_id"))) \
    .filter(col("fecha").isNotNull() & col("estacion_id").isNotNull() &
            col("int_viento").isNotNull() & col("dir_viento").isNotNull()) \
    .dropDuplicates() \
    .filter((col("int_viento") >= 0) & (col("int_viento") <= 200) &
            (col("dir_viento") >= 0) & (col("dir_viento") <= 360))
print(f"  Registros finales: {df_viento_rfn.count():,}")

print("\n=== LIMPIEZA - PRECIPITACION ===")
df_lluvia_rfn = limpiar(df_lluvia_raw, "precip_horaria", 0, 300)

print("\n=== LIMPIEZA - HUMEDAD ===")
df_humedad_rfn = limpiar(df_humedad_raw, col_valor_humedad, 0, 100)

print("\n=== LIMPIEZA - PRESION ===")
df_presion_rfn = limpiar(df_presion_raw, col_valor_presion, 900, 1100)

print("\n=== LIMPIEZA - HELIOFANIA ===")
df_helio_rfn = limpiar(df_helio_raw, col_valor_helio, 0, 24)

In [ ]:
# Guardar en /rfn como parquet
RFN = "hdfs://localhost:9000/rfn/obligatorio"

df_temp_rfn.write.mode("overwrite").parquet(f"{RFN}/temperatura")
df_viento_rfn.write.mode("overwrite").parquet(f"{RFN}/viento")
df_lluvia_rfn.write.mode("overwrite").parquet(f"{RFN}/precipitacion")
df_humedad_rfn.write.mode("overwrite").parquet(f"{RFN}/humedad")
df_presion_rfn.write.mode("overwrite").parquet(f"{RFN}/presion")
df_helio_rfn.write.mode("overwrite").parquet(f"{RFN}/heliofania")

print("Datos refinados guardados en /rfn correctamente")

## 11. Verificacion final — comparacion lnd vs rfn

In [ ]:
tablas_rfn = {
    "temperatura":   df_temp_rfn,
    "viento":        df_viento_rfn,
    "precipitacion": df_lluvia_rfn,
    "humedad":       df_humedad_rfn,
    "presion":       df_presion_rfn,
    "heliofania":    df_helio_rfn,
}

print(f"{'Tabla':<15} {'LND (crudos)':>14} {'RFN (limpios)':>14} {'Eliminados':>12}")
print("-"*58)

for nombre in tablas_rfn:
    n_lnd = tablas[nombre].count()
    n_rfn = tablas_rfn[nombre].count()
    eliminados = n_lnd - n_rfn
    pct = eliminados / n_lnd * 100 if n_lnd > 0 else 0
    print(f"{nombre:<15} {n_lnd:>14,} {n_rfn:>14,} {eliminados:>10,} ({pct:.1f}%)")

In [ ]:
spark.stop()
print("Sesion Spark cerrada.")